# Phase 9: Statistical Analysis & Hypothesis Testing
## Advanced Statistical Methods and Hypothesis Validation
**Objective:** Perform statistical tests to validate hypotheses and assumptions
**Output:** Statistical insights for data-driven decision making

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency, pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

print(f"Dataset loaded: {df.shape}")
print(f"\nBasic Statistics:")
print(df[['rating', 'num_ratings', 'avg_cost']].describe())

Dataset loaded: (7105, 12)

Basic Statistics:
            rating   num_ratings     avg_cost
count  7105.000000   7105.000000  7105.000000
mean      3.514117    188.921042   539.161013
std       0.461028    592.171049   461.211329
min       1.800000      1.000000    40.000000
25%       3.200000     16.000000   300.000000
50%       3.500000     40.000000   400.000000
75%       3.800000    128.000000   600.000000
max       4.900000  16345.000000  6000.000000


In [3]:
# 1. Normality Tests
print("\n" + "="*70)
print("1. NORMALITY TESTS (Shapiro-Wilk)")
print("="*70)

# Test sample (Shapiro-Wilk doesn't work well with large datasets)
sample_size = min(5000, len(df))
sample = df.sample(sample_size, random_state=42)

for col in ['rating', 'num_ratings', 'avg_cost']:
    stat, p_value = stats.shapiro(sample[col])
    print(f"\n{col}:")
    print(f"  Test Statistic: {stat:.4f}")
    print(f"  P-value: {p_value:.4e}")
    print(f"  Result: {'Normal' if p_value > 0.05 else 'Not Normal'} (α=0.05)")


1. NORMALITY TESTS (Shapiro-Wilk)

rating:
  Test Statistic: 0.9839
  P-value: 2.0663e-23
  Result: Not Normal (α=0.05)

num_ratings:
  Test Statistic: 0.2803
  P-value: 3.2050e-88
  Result: Not Normal (α=0.05)

avg_cost:
  Test Statistic: 0.7057
  P-value: 4.1673e-69
  Result: Not Normal (α=0.05)


In [4]:
# 2. Hypothesis Test: Online Order Impact on Ratings
print("\n" + "="*70)
print("2. HYPOTHESIS TEST: Online Order Impact on Rating")
print("="*70)

with_online = df[df['has_online_order'] == 1]['rating']
without_online = df[df['has_online_order'] == 0]['rating']

# T-test
t_stat, t_pvalue = ttest_ind(with_online, without_online)

# Mann-Whitney U test (non-parametric alternative)
u_stat, u_pvalue = mannwhitneyu(with_online, without_online)

print(f"\nWith Online Order - Mean Rating: {with_online.mean():.4f}")
print(f"Without Online Order - Mean Rating: {without_online.mean():.4f}")
print(f"Difference: {with_online.mean() - without_online.mean():.4f}")

print(f"\nIndependent t-test:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {t_pvalue:.4e}")
print(f"  Conclusion: {'Significant difference' if t_pvalue < 0.05 else 'No significant difference'} (α=0.05)")

print(f"\nMann-Whitney U test (non-parametric):")
print(f"  U-statistic: {u_stat:.4f}")
print(f"  p-value: {u_pvalue:.4e}")
print(f"  Conclusion: {'Significant difference' if u_pvalue < 0.05 else 'No significant difference'} (α=0.05)")


2. HYPOTHESIS TEST: Online Order Impact on Rating

With Online Order - Mean Rating: 3.5519
Without Online Order - Mean Rating: 3.4724
Difference: 0.0795

Independent t-test:
  t-statistic: 7.2810
  p-value: 3.6700e-13
  Conclusion: Significant difference (α=0.05)

Mann-Whitney U test (non-parametric):
  U-statistic: 7030647.5000
  p-value: 1.3130e-17
  Conclusion: Significant difference (α=0.05)


In [5]:
# 3. Hypothesis Test: Table Booking Impact on Ratings
print("\n" + "="*70)
print("3. HYPOTHESIS TEST: Table Booking Impact on Rating")
print("="*70)

with_booking = df[df['has_table_booking'] == 1]['rating']
without_booking = df[df['has_table_booking'] == 0]['rating']

t_stat, t_pvalue = ttest_ind(with_booking, without_booking)
u_stat, u_pvalue = mannwhitneyu(with_booking, without_booking)

print(f"\nWith Table Booking - Mean Rating: {with_booking.mean():.4f}")
print(f"Without Table Booking - Mean Rating: {without_booking.mean():.4f}")
print(f"Difference: {with_booking.mean() - without_booking.mean():.4f}")

print(f"\nIndependent t-test:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {t_pvalue:.4e}")
print(f"  Conclusion: {'Significant difference' if t_pvalue < 0.05 else 'No significant difference'} (α=0.05)")


3. HYPOTHESIS TEST: Table Booking Impact on Rating

With Table Booking - Mean Rating: 4.0360
Without Table Booking - Mean Rating: 3.4531
Difference: 0.5829

Independent t-test:
  t-statistic: 35.3921
  p-value: 7.5712e-253
  Conclusion: Significant difference (α=0.05)


In [6]:
# 4. Correlation Analysis
print("\n" + "="*70)
print("4. CORRELATION ANALYSIS")
print("="*70)

numeric_cols = ['rating', 'num_ratings', 'avg_cost', 'has_online_order', 'has_table_booking']

# Pearson Correlation
print("\nPearson Correlation:")
pearson_corr = df[numeric_cols].corr()
print(pearson_corr['rating'].sort_values(ascending=False))

# Detailed correlation with p-values
print("\n\nDetailed Correlation with Rating:")
for col in numeric_cols:
    if col != 'rating':
        corr, p_value = pearsonr(df['rating'], df[col])
        print(f"\n{col}:")
        print(f"  Correlation coefficient: {corr:.4f}")
        print(f"  p-value: {p_value:.4e}")
        print(f"  Significance: {'Significant' if p_value < 0.05 else 'Not significant'} (α=0.05)")


4. CORRELATION ANALYSIS

Pearson Correlation:
rating               1.000000
has_table_booking    0.387185
num_ratings          0.380215
avg_cost             0.374133
has_online_order     0.086071
Name: rating, dtype: float64


Detailed Correlation with Rating:

num_ratings:
  Correlation coefficient: 0.3802
  p-value: 3.6431e-243
  Significance: Significant (α=0.05)

avg_cost:
  Correlation coefficient: 0.3741
  p-value: 6.5721e-235
  Significance: Significant (α=0.05)

has_online_order:
  Correlation coefficient: 0.0861
  p-value: 3.6700e-13
  Significance: Significant (α=0.05)

has_table_booking:
  Correlation coefficient: 0.3872
  p-value: 7.5712e-253
  Significance: Significant (α=0.05)


In [7]:
# 5. Chi-Square Test: Rating Category vs Online Order
print("\n" + "="*70)
print("5. CHI-SQUARE TEST: Rating Category vs Online Order")
print("="*70)

df['rating_category'] = pd.cut(df['rating'], bins=[0, 3, 3.5, 4, 4.5, 5], 
                                 labels=['Poor', 'Average', 'Good', 'Very Good', 'Excellent'],
                                 include_lowest=True)

contingency = pd.crosstab(df['rating_category'], df['has_online_order'])
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"\nContingency Table:")
print(contingency)

print(f"\nChi-Square Test Results:")
print(f"  Chi-square statistic: {chi2:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Degrees of freedom: {dof}")
print(f"  Conclusion: {'Significant association' if p_value < 0.05 else 'No significant association'} (α=0.05)")


5. CHI-SQUARE TEST: Rating Category vs Online Order

Contingency Table:
has_online_order     0     1
rating_category             
Poor               671   614
Average           1285  1057
Good              1022  1584
Very Good          369   452
Excellent           31    20

Chi-Square Test Results:
  Chi-square statistic: 139.8816
  p-value: 2.9922e-29
  Degrees of freedom: 4
  Conclusion: Significant association (α=0.05)


In [8]:
# 6. ANOVA: Rating across Restaurant Types (Top 5)
print("\n" + "="*70)
print("6. ANOVA: Rating Across Top 5 Restaurant Types")
print("="*70)

top_5_types = df['restaurant_type'].value_counts().head(5).index
groups = [df[df['restaurant_type'] == rtype]['rating'].values for rtype in top_5_types]

f_stat, p_value = stats.f_oneway(*groups)

print(f"\nMean Ratings by Restaurant Type (Top 5):")
for rtype in top_5_types:
    mean_rating = df[df['restaurant_type'] == rtype]['rating'].mean()
    print(f"  {rtype}: {mean_rating:.4f}")

print(f"\nANOVA Results:")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Conclusion: {'Significant difference' if p_value < 0.05 else 'No significant difference'} (α=0.05)")


6. ANOVA: Rating Across Top 5 Restaurant Types

Mean Ratings by Restaurant Type (Top 5):
  Quick Bites: 3.4023
  Casual Dining: 3.5946
  Cafe: 3.6911
  Delivery: 3.3830
  Takeaway, Delivery: 3.2913

ANOVA Results:
  F-statistic: 94.0397
  p-value: 1.6964e-77
  Conclusion: Significant difference (α=0.05)


In [9]:
# 7. Effect Size Analysis
print("\n" + "="*70)
print("7. EFFECT SIZE ANALYSIS (Cohen's d)")
print("="*70)

def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

with_online = df[df['has_online_order'] == 1]['rating']
without_online = df[df['has_online_order'] == 0]['rating']
d = cohens_d(with_online, without_online)

print(f"\nOnline Order Impact (Cohen's d): {d:.4f}")
if abs(d) < 0.2:
    effect = "Negligible"
elif abs(d) < 0.5:
    effect = "Small"
elif abs(d) < 0.8:
    effect = "Medium"
else:
    effect = "Large"
print(f"Effect Size: {effect}")


7. EFFECT SIZE ANALYSIS (Cohen's d)

Online Order Impact (Cohen's d): 0.1730
Effect Size: Negligible


In [10]:
# 8. Statistical Summary
print("\n" + "="*70)
print("STATISTICAL SUMMARY & KEY FINDINGS")
print("="*70)

print(f"\n1. Data Distribution:")
print(f"   - Rating: {'Normal' if stats.shapiro(sample['rating'])[1] > 0.05 else 'Not Normal'} distribution")
print(f"   - Skewness: {stats.skew(df['rating']):.4f}")
print(f"   - Kurtosis: {stats.kurtosis(df['rating']):.4f}")

print(f"\n2. Key Relationships:")
print(f"   - Online Order: {'Significantly' if t_pvalue < 0.05 else 'Not significantly'} affects rating")
print(f"   - Strongest correlation: {pearson_corr['rating'].drop('rating').abs().idxmax()} ({pearson_corr['rating'].drop('rating').abs().max():.3f})")

print(f"\n✓ Statistical Analysis Complete!")


STATISTICAL SUMMARY & KEY FINDINGS

1. Data Distribution:
   - Rating: Not Normal distribution
   - Skewness: -0.0623
   - Kurtosis: -0.5169

2. Key Relationships:
   - Online Order: Significantly affects rating
   - Strongest correlation: has_table_booking (0.387)

✓ Statistical Analysis Complete!
